<a href="https://colab.research.google.com/github/Benhubbard9891/Mnemonik/blob/main/mnemonik_v31_colab_verify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MNĒMONIKEVOKE v3.1 (FTS5) — Colab Verification Run

This notebook verifies the **fixed** MNĒMONIKEVOKE v3.1 asynchronous memory engine — the exact code and the exact test suite that pass locally (11/11) — in a clean Colab environment.

**What it does (Runtime → Run all):**

1. Installs `aiosqlite` (the one dependency not in Colab's default environment; `pydantic` already is). No OpenAI key is needed — the suite drives the engine with its deterministic mock embedder/LLM.
2. Writes the fixed engine to `mnemonik_v31_full.py` — byte-for-byte the local fixed copy (fusion calibration: graph folded into RRF as a third ranked list with per-signal `fusion_weights=(1.0, 1.0, 0.5)`, negative weights raise; N+1 fix: one batched vector fetch per recall; embedding-dimension guard raises on mismatch).
3. Writes the test suite to `mnemonik_v31_tests_colab.py` — identical to the local suite except one line: `MODULE_PATH` points at the notebook's working directory instead of the local absolute path.
4. Runs the suite and prints one summary line.

**Expected final line:**

```
MNĒMONIKEVOKE v3.1 COLAB VERIFICATION: PASS — 11/11 tests passed
```

The 11 tests cover: ingest/active facts, duplicate no-op, supersession, fused-score bounds under a 200-edge graph, fusion-weight plumbing (zero graph weight; negative weight raises), recall result shape/limit, exactly one vectors query per recall (N+1 fix), dimension-mismatch raising, FTS5/BM25 sparse hit, and half-life decay + reinforcement.

In [ ]:
# aiosqlite is NOT in Colab's default environment; pydantic is.
# openai is not needed: the suite uses the engine's deterministic mocks.
!pip install -q aiosqlite

In [ ]:
%%writefile mnemonik_v31_full.py
"""
MNĒMONIKEVOKE v3.1 - Asynchronous Agent Memory Model (FTS5 Edition)
===================================================================

Fully non-blocking agent-memory engine with:
- aiosqlite for asynchronous database operations
- SQLite FTS5 native extension for high-performance BM25 sparse retrieval
- AsyncOpenAI for non-blocking embeddings and LLM extraction
- Pydantic structured outputs for deterministic JSON schema enforcement
- Batched embedding generation for index rebuilding
- Hybrid retrieval (Dense + Sparse/BM25 + Graph + RRF)
- Bi-temporal validity and supersession tracking

License: MIT / Open Domain
"""

from __future__ import annotations

import asyncio
import hashlib
import json
import math
import os
import re
from dataclasses import dataclass
from typing import Iterable, Literal, Optional

import aiosqlite
from pydantic import BaseModel, Field

DAY = 86_400.0

# ============================================================================
# 1. DATA MODELS
# ============================================================================

@dataclass
class Fact:
    id: str
    subject: str
    predicate: str
    object: str
    scope: Optional[str] = None
    valid_from: Optional[float] = None
    valid_to: Optional[float] = None
    supersedes: Optional[str] = None
    created_at: float = 0.0
    strength: float = 1.0
    last_accessed: float = 0.0
    provenance: str = ""

@dataclass
class RecallResult:
    fact: Fact
    dense_score: float
    sparse_score: float
    fused_score: float
    graph_score: float = 0.0

class ExtractedFact(BaseModel):
    subject: str = Field(description="The primary entity (e.g., 'Rust', 'User')")
    predicate: str = Field(description="The relationship (e.g., 'uses', 'likes')")
    object: str = Field(description="The target entity or concept (e.g., 'backend', 'pizza')")
    scope: Optional[str] = Field(default=None, description="Domain context if applicable")

class ExtractionResponse(BaseModel):
    facts: list[ExtractedFact]

class ReconciliationDecision(BaseModel):
    decision: Literal["ADD", "UPDATE", "SUPERSEDE", "MERGE", "NOOP"]
    target_id: Optional[str] = Field(default=None, description="The ID of the existing fact, if applicable")

# ============================================================================
# 2. ASYNC INTERFACES
# ============================================================================

class Clock:
    def __init__(self, t0: float = 0.0):
        self._now = float(t0)

    def now(self) -> float:
        return self._now

    def advance(self, delta: float) -> None:
        if delta < 0:
            raise ValueError("Time cannot flow backwards.")
        self._now += float(delta)

class AsyncEmbedder:
    async def embed(self, text: str) -> list[float]:
        raise NotImplementedError

    async def embed_batch(self, texts: list[str]) -> list[list[float]]:
        raise NotImplementedError

class AsyncLLMClient:
    async def extract_triplets(self, text: str) -> list[dict]:
        raise NotImplementedError

    async def reconcile(
        self,
        facts: list[Fact],
        new_fact: dict,
    ) -> tuple[str, Optional[str], Optional[list[str]]]:
        raise NotImplementedError


# ============================================================================
# 3. PRODUCTION PROVIDERS (OpenAI)
# ============================================================================

class AsyncOpenAIEmbedder(AsyncEmbedder):
    def __init__(self, api_key: str, model: str = "text-embedding-3-small"):
        from openai import AsyncOpenAI
        self.client = AsyncOpenAI(api_key=api_key)
        self.model = model

    async def embed(self, text: str) -> list[float]:
        results = await self.embed_batch([text])
        return results[0]

    async def embed_batch(self, texts: list[str]) -> list[list[float]]:
        if not texts:
            return []
        clean_texts = [text.replace("\n", " ") for text in texts]
        response = await self.client.embeddings.create(
            input=clean_texts,
            model=self.model
        )
        return [data.embedding for data in response.data]

class AsyncOpenAILLM(AsyncLLMClient):
    def __init__(self, api_key: str, model: str = "gpt-4o-mini"):
        from openai import AsyncOpenAI
        self.client = AsyncOpenAI(api_key=api_key)
        self.model = model

    async def extract_triplets(self, text: str) -> list[dict]:
        completion = await self.client.beta.chat.completions.parse(
            model=self.model,
            messages=[
                {"role": "system", "content": "Extract factual triplets from the user's input."},
                {"role": "user", "content": text}
            ],
            response_format=ExtractionResponse,
        )
        if not completion.choices[0].message.parsed:
            return []
        return [fact.model_dump() for fact in completion.choices[0].message.parsed.facts]

    async def reconcile(
        self,
        facts: list[Fact],
        new_fact: dict,
    ) -> tuple[str, Optional[str], Optional[list[str]]]:
        if not facts:
            return "ADD", None, None

        existing_context = "\n".join(
            f"ID: {f.id} | {f.subject} {f.predicate} {f.object} (Scope: {f.scope})"
            for f in facts
        )

        prompt = f"""
        Compare the NEW FACT against the EXISTING FACTS.
        Decide the correct operation:
        - ADD: Completely new information.
        - SUPERSEDE: Replaces an existing fact.
        - UPDATE: Modifies the details of an existing fact.
        - MERGE: Identical meaning to an existing fact; reinforce it.
        - NOOP: Redundant or irrelevant.

        EXISTING FACTS:
        {existing_context}

        NEW FACT:
        {new_fact['subject']} {new_fact['predicate']} {new_fact['object']} (Scope: {new_fact.get('scope')})
        """

        completion = await self.client.beta.chat.completions.parse(
            model=self.model,
            messages=[
                {"role": "system", "content": "You are a bi-temporal database reconciliation engine."},
                {"role": "user", "content": prompt}
            ],
            response_format=ReconciliationDecision,
        )

        parsed = completion.choices[0].message.parsed
        if not parsed:
            return "ADD", None, None
        return parsed.decision, parsed.target_id, None


# ============================================================================
# 4. DETERMINISTIC MOCKS (For local testing without API keys)
# ============================================================================

class AsyncHashingEmbedder(AsyncEmbedder):
    def __init__(self, dim: int = 64):
        self.dim = dim

    async def embed(self, text: str) -> list[float]:
        clean = text.lower().strip()
        vector: list[float] = []
        for index in range(self.dim):
            digest = hashlib.sha256(f"{clean}|dimension:{index}".encode("utf-8")).digest()
            raw = int.from_bytes(digest[:8], "big")
            normalized = raw / ((1 << 64) - 1)
            vector.append((normalized * 2.0) - 1.0)
        norm = math.sqrt(sum(value * value for value in vector))
        if norm <= 1e-12: return [0.0] * self.dim
        return [value / norm for value in vector]

    async def embed_batch(self, texts: list[str]) -> list[list[float]]:
        return [await self.embed(text) for text in texts]

class AsyncHeuristicLLM(AsyncLLMClient):
    async def extract_triplets(self, text: str) -> list[dict]:
        clean = text.strip().rstrip(".")

        migration = re.search(r"(?:we are|we're)\s+(?:shifting|moving|transitioning)\s+(?:the\s+)?(?:entire\s+)?([\w-]+)\s+from\s+([\w-]+)\s+to\s+([\w-]+)", clean, re.IGNORECASE)
        if migration:
            scope, _from_value, to_value = migration.groups()
            return [{"subject": to_value, "predicate": "uses", "object": scope, "scope": scope.lower()}]

        uses = re.search(r"\bI\s+use\s+([\w-]+)\s+for\s+(?:the\s+)?(.+)", clean, re.IGNORECASE)
        if uses:
            subject, object_value = uses.groups()
            scope = "backend" if "backend" in object_value.lower() else "general"
            return [{"subject": subject, "predicate": "uses", "object": object_value.strip(), "scope": scope}]

        return []

    async def reconcile(self, facts: list[Fact], new_fact: dict) -> tuple[str, Optional[str], Optional[list[str]]]:
        ns, np, no, nscope = new_fact.get("subject"), new_fact.get("predicate"), new_fact.get("object"), new_fact.get("scope")
        for fact in facts:
            if fact.predicate == np and fact.scope == nscope and nscope is not None:
                if fact.subject != ns and fact.object == no: return "SUPERSEDE", fact.id, None
                if fact.subject == ns and fact.object != no: return "UPDATE", fact.id, None
                if fact.subject == ns and fact.object == no: return "NOOP", fact.id, None
        return "ADD", None, None


# ============================================================================
# 5. CORE ASYNC ENGINE (NOW WITH FTS5)
# ============================================================================

class MnēmonikEvokeAsync:
    def __init__(
        self,
        clock: Clock,
        embedder: AsyncEmbedder,
        llm: AsyncLLMClient,
        forgetting_half_life: float = 7 * DAY,
        recall_reinforcement: float = 0.10,
    ):
        self.clock = clock
        self.embedder = embedder
        self.llm = llm
        self.forgetting_half_life = forgetting_half_life
        self.recall_reinforcement = recall_reinforcement
        self.db: Optional[aiosqlite.Connection] = None

    async def setup(self) -> MnēmonikEvokeAsync:
        self.db = await aiosqlite.connect(":memory:")
        self.db.row_factory = aiosqlite.Row
        await self._init_db()
        return self

    async def close(self) -> None:
        if self.db:
            await self.db.close()

    async def _init_db(self) -> None:
        if not self.db: return
        await self.db.executescript(
            """
            CREATE TABLE facts (
                id TEXT PRIMARY KEY,
                subject TEXT NOT NULL,
                predicate TEXT NOT NULL,
                object TEXT NOT NULL,
                scope TEXT,
                valid_from REAL,
                valid_to REAL,
                supersedes TEXT,
                created_at REAL NOT NULL,
                strength REAL NOT NULL,
                last_accessed REAL NOT NULL,
                provenance TEXT NOT NULL
            );

            CREATE TABLE vectors (
                fact_id TEXT PRIMARY KEY,
                vector BLOB NOT NULL,
                FOREIGN KEY(fact_id) REFERENCES facts(id)
            );

            -- Replacing manual term frequency table with SQLite's native FTS5 extension!
            CREATE VIRTUAL TABLE facts_fts USING fts5(
                fact_id UNINDEXED,
                text
            );

            CREATE TABLE edges (
                source TEXT NOT NULL,
                target TEXT NOT NULL,
                predicate TEXT NOT NULL,
                weight REAL NOT NULL,
                PRIMARY KEY(source, target, predicate)
            );

            CREATE INDEX facts_predicate_idx ON facts(predicate);
            CREATE INDEX facts_scope_idx ON facts(scope);
            CREATE INDEX facts_validity_idx ON facts(valid_to);
            """
        )
        await self.db.commit()

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------
    @staticmethod
    def _fact_text(fact: Fact) -> str:
        return f"{fact.subject} {fact.predicate} {fact.object}"

    @staticmethod
    def _tokenize(text: str) -> list[str]:
        # Extract alphanumeric tokens to prevent FTS5 syntax errors
        return re.findall(r"\w+", text.lower())

    @staticmethod
    def _row_to_fact(row: aiosqlite.Row) -> Fact:
        return Fact(
            id=row["id"], subject=row["subject"], predicate=row["predicate"],
            object=row["object"], scope=row["scope"], valid_from=row["valid_from"],
            valid_to=row["valid_to"], supersedes=row["supersedes"], created_at=row["created_at"],
            strength=row["strength"], last_accessed=row["last_accessed"], provenance=row["provenance"],
        )

    def _normalize_raw_fact(self, raw: dict) -> dict:
        for key in ("subject", "predicate", "object"):
            if not raw.get(key):
                raise ValueError(f"Extracted fact missing '{key}'.")
        return {
            "subject": str(raw["subject"]).strip(),
            "predicate": str(raw["predicate"]).strip(),
            "object": str(raw["object"]).strip(),
            "scope": str(raw["scope"]).strip() if raw.get("scope") else None,
        }

    # ------------------------------------------------------------------
    # Write pipeline
    # ------------------------------------------------------------------
    async def observe(self, text: str, session: str = "default") -> list[tuple[str, Fact]]:
        extracted = await self.llm.extract_triplets(text)
        results: list[tuple[str, Fact]] = []
        now = self.clock.now()

        for raw in extracted:
            normalized = self._normalize_raw_fact(raw)
            existing = await self.get_active_facts(predicate=normalized["predicate"])

            decision, target_id, _ = await self.llm.reconcile(existing, normalized)

            digest = hashlib.sha256(json.dumps({"raw": normalized, "time": now}, sort_keys=True).encode()).hexdigest()[:16]
            fact_id = f"fact_{digest}"

            proposed = Fact(
                id=fact_id, subject=normalized["subject"], predicate=normalized["predicate"],
                object=normalized["object"], scope=normalized.get("scope"), valid_from=now,
                valid_to=None, supersedes=(target_id if decision == "SUPERSEDE" else None),
                created_at=now, strength=1.0, last_accessed=now, provenance=f"session:{session}|raw:{text}"
            )

            if decision == "ADD":
                await self.insert_fact(proposed)
                results.append(("ADD", proposed))
            elif decision == "SUPERSEDE" and target_id:
                await self.update_fact_validity(target_id, now)
                await self.insert_fact(proposed)
                results.append(("SUPERSEDE", proposed))
            elif decision == "UPDATE" and target_id:
                persisted = await self.update_fact_content(target_id, proposed)
                results.append(("UPDATE", persisted))
            elif decision == "MERGE" and target_id:
                persisted = await self.merge_fact(target_id, proposed)
                results.append(("MERGE", persisted))
            else:
                existing_fact = await self.get_fact(target_id) if target_id else None
                results.append(("NOOP", existing_fact or proposed))

        await self._rebuild_indexes()
        if self.db: await self.db.commit()
        return results

    # ------------------------------------------------------------------
    # Fact persistence (Async)
    # ------------------------------------------------------------------
    async def insert_fact(self, fact: Fact) -> None:
        if not self.db: return
        await self.db.execute(
            """INSERT INTO facts (id, subject, predicate, object, scope, valid_from, valid_to,
               supersedes, created_at, strength, last_accessed, provenance)
               VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)""",
            (fact.id, fact.subject, fact.predicate, fact.object, fact.scope, fact.valid_from,
             fact.valid_to, fact.supersedes, fact.created_at, fact.strength, fact.last_accessed, fact.provenance)
        )

    async def update_fact_validity(self, fact_id: str, valid_to: float) -> None:
        if not self.db: return
        await self.db.execute("UPDATE facts SET valid_to=? WHERE id=?", (valid_to, fact_id))

    async def update_fact_content(self, fact_id: str, replacement: Fact) -> Fact:
        if not self.db: raise RuntimeError("No DB connection")
        await self.db.execute(
            """UPDATE facts SET subject=?, predicate=?, object=?, scope=?, valid_from=?,
               valid_to=?, supersedes=?, provenance=? WHERE id=?""",
            (replacement.subject, replacement.predicate, replacement.object, replacement.scope,
             replacement.valid_from, replacement.valid_to, replacement.supersedes, replacement.provenance, fact_id)
        )
        persisted = await self.get_fact(fact_id)
        if not persisted: raise RuntimeError("Fact disappeared during update.")
        return persisted

    async def merge_fact(self, fact_id: str, incoming: Fact) -> Fact:
        current = await self.get_fact(fact_id)
        if current is None: raise KeyError(f"Unknown fact: {fact_id}")

        provenance = "\n".join(dict.fromkeys(v for v in (current.provenance, incoming.provenance) if v))
        if self.db:
            await self.db.execute(
                "UPDATE facts SET provenance=?, strength=?, last_accessed=? WHERE id=?",
                (provenance, max(current.strength, incoming.strength), self.clock.now(), fact_id)
            )
        persisted = await self.get_fact(fact_id)
        if not persisted: raise RuntimeError("Fact disappeared during merge.")
        return persisted

    # ------------------------------------------------------------------
    # Queries (Async)
    # ------------------------------------------------------------------
    async def get_fact(self, fact_id: str) -> Optional[Fact]:
        if not self.db: return None
        async with self.db.execute("SELECT * FROM facts WHERE id=?", (fact_id,)) as cursor:
            row = await cursor.fetchone()
        return self._row_to_fact(row) if row else None

    async def get_all_facts(self) -> list[Fact]:
        if not self.db: return []
        async with self.db.execute("SELECT * FROM facts ORDER BY created_at, id") as cursor:
            rows = await cursor.fetchall()
        return [self._row_to_fact(row) for row in rows]

    async def get_active_facts(self, predicate: Optional[str] = None, scope: Optional[str] = None, at: Optional[float] = None) -> list[Fact]:
        if not self.db: return []
        timestamp = self.clock.now() if at is None else at
        clauses = ["(valid_from IS NULL OR valid_from <= ?)", "(valid_to IS NULL OR valid_to > ?)"]
        params: list[object] = [timestamp, timestamp]

        if predicate: clauses.append("predicate=?"); params.append(predicate)
        if scope: clauses.append("scope=?"); params.append(scope)

        query = f"SELECT * FROM facts WHERE {' AND '.join(clauses)} ORDER BY created_at, id"
        async with self.db.execute(query, params) as cursor:
            rows = await cursor.fetchall()
        return [self._row_to_fact(row) for row in rows]

    # ------------------------------------------------------------------
    # Indexes (Batched Async with FTS5)
    # ------------------------------------------------------------------
    async def _rebuild_indexes(self) -> None:
        if not self.db: return
        await self.db.execute("DELETE FROM vectors")
        await self.db.execute("DELETE FROM facts_fts")
        await self.db.execute("DELETE FROM edges")

        all_facts = await self.get_all_facts()
        if not all_facts: return

        texts = [self._fact_text(f) for f in all_facts]
        vectors = await self.embedder.embed_batch(texts)

        for fact, text, vector in zip(all_facts, texts, vectors):
            # 1. Update dense vector
            await self.db.execute(
                "INSERT INTO vectors(fact_id, vector) VALUES (?, ?)",
                (fact.id, json.dumps(vector, separators=(",", ":")).encode("utf-8"))
            )

            # 2. Update FTS5 virtual table
            await self.db.execute(
                "INSERT INTO facts_fts(fact_id, text) VALUES (?, ?)",
                (fact.id, text)
            )

            # 3. Update knowledge graph edges
            await self.db.execute(
                """INSERT INTO edges(source, target, predicate, weight) VALUES (?, ?, ?, ?)
                   ON CONFLICT(source, target, predicate) DO UPDATE SET weight=edges.weight + excluded.weight""",
                (fact.subject, fact.object, fact.predicate, 1.0)
            )

    # ------------------------------------------------------------------
    # Retrieval & Ranking (Async)
    # ------------------------------------------------------------------
    async def _dense_rank(self, query: str, candidates: list[Fact]) -> list[tuple[Fact, float]]:
        if not self.db or not candidates: return []
        query_vector = await self.embedder.embed(query)
        query_dim = len(query_vector)
        query_norm = math.sqrt(sum(v * v for v in query_vector))

        # One batched fetch for all candidates — not one round-trip per fact.
        placeholders = ",".join("?" for _ in candidates)
        async with self.db.execute(
            f"SELECT fact_id, vector FROM vectors WHERE fact_id IN ({placeholders})",
            [fact.id for fact in candidates],
        ) as cursor:
            rows = await cursor.fetchall()
        vectors = {row["fact_id"]: row["vector"] for row in rows}

        scored: list[tuple[Fact, float]] = []
        for fact in candidates:
            blob = vectors.get(fact.id)
            if blob is None: continue

            stored_vector = json.loads(bytes(blob).decode("utf-8"))
            if len(stored_vector) != query_dim:
                raise ValueError(
                    f"Embedding dimension mismatch for fact {fact.id}: "
                    f"stored={len(stored_vector)}, query={query_dim}"
                )
            right_norm = math.sqrt(sum(v * v for v in stored_vector))
            if query_norm <= 1e-12 or right_norm <= 1e-12:
                score = 0.0
            else:
                dot = sum(l * r for l, r in zip(query_vector, stored_vector))
                score = dot / (query_norm * right_norm)

            scored.append((fact, score))
        return sorted(scored, key=lambda item: (item[1], item[0].strength), reverse=True)

    async def _sparse_rank(self, query: str, candidates: list[Fact]) -> list[tuple[Fact, float]]:
        """Powered by SQLite FTS5 BM25 scoring."""
        if not self.db: return []

        tokens = self._tokenize(query)
        if not tokens:
            return []

        # Create a simple OR query string for FTS5 (e.g. "token1 OR token2")
        fts_query = " OR ".join(tokens)

        # SQLite FTS5 bm25() returns negative values where more negative is better.
        # We negate it so higher = better, aligning with dense and graph scores.
        async with self.db.execute(
            "SELECT fact_id, -bm25(facts_fts) as score FROM facts_fts WHERE facts_fts MATCH ?",
            (fts_query,)
        ) as cursor:
            rows = await cursor.fetchall()

        scores = {row["fact_id"]: float(row["score"]) for row in rows}
        candidate_ids = {fact.id for fact in candidates}

        scored: list[tuple[Fact, float]] = []
        for fact in candidates:
            if fact.id in scores:
                scored.append((fact, scores[fact.id]))

        return sorted(scored, key=lambda item: (item[1], item[0].strength), reverse=True)

    async def _graph_expansion(self, seeds: Iterable[Fact], candidates: list[Fact], hops: int = 1) -> dict[str, float]:
        if not self.db or hops <= 0: return {}
        entity_frontier = {f.subject for f in seeds}.union({f.object for f in seeds})
        scores: dict[str, float] = {}
        visited: set[str] = set()

        for _ in range(hops):
            next_frontier: set[str] = set()
            for entity in entity_frontier:
                if entity in visited: continue
                visited.add(entity)

                async with self.db.execute("SELECT source, target, weight FROM edges WHERE source=? OR target=?", (entity, entity)) as cursor:
                    rows = await cursor.fetchall()

                for row in rows:
                    source, target, weight = row["source"], row["target"], float(row["weight"])
                    for fact in candidates:
                        if fact.subject in (source, target) or fact.object in (source, target):
                            scores[fact.id] = scores.get(fact.id, 0.0) + weight
                    next_frontier.update([source, target])
            entity_frontier = next_frontier
        return scores

    async def recall(
        self,
        query: str,
        limit: int = 5,
        include_history: bool = False,
        graph_hops: int = 1,
        reinforce: bool = True,
        fusion_weights: tuple[float, float, float] = (1.0, 1.0, 0.5),
    ) -> list[RecallResult]:
        if limit <= 0: return []
        candidates = await self.get_all_facts() if include_history else await self.get_active_facts()
        if not candidates: return []

        dense = await self._dense_rank(query, candidates)
        sparse = await self._sparse_rank(query, candidates)

        dense_ranks = {fact.id: r for r, (fact, _) in enumerate(dense, 1)}
        sparse_ranks = {fact.id: r for r, (fact, _) in enumerate(sparse, 1)}
        dense_scores = {fact.id: s for fact, s in dense}
        sparse_scores = {fact.id: s for fact, s in sparse}

        seeds = [fact for fact, _ in dense[:3]]
        graph_scores = await self._graph_expansion(seeds, candidates, hops=graph_hops)
        # Rank the graph signal so it fuses on the same scale as dense/sparse.
        # Raw graph scores are unbounded weight sums; ranks are not.
        graph_ranks = {
            fact_id: rank
            for rank, (fact_id, _) in enumerate(
                sorted(graph_scores.items(), key=lambda item: item[1], reverse=True), 1
            )
        }

        rrf_k = 60.0
        missing_rank = 10_000  # penalty rank for candidates absent from a signal
        w_dense, w_sparse, w_graph = fusion_weights
        if w_dense < 0 or w_sparse < 0 or w_graph < 0:
            raise ValueError(f"fusion_weights must be non-negative, got {fusion_weights!r}")
        results: list[RecallResult] = []

        for fact in candidates:
            # Weighted Reciprocal Rank Fusion over all three signals. Every
            # signal is rank-based, so each contributes at most
            # weight/(rrf_k + 1) and no single signal can dominate by raw score
            # magnitude. Graph defaults to half weight: it expands from dense
            # seeds, so it partially double-counts the dense signal.
            fused = (
                w_dense / (rrf_k + dense_ranks.get(fact.id, missing_rank))
                + w_sparse / (rrf_k + sparse_ranks.get(fact.id, missing_rank))
                + w_graph / (rrf_k + graph_ranks.get(fact.id, missing_rank))
            )

            strength_factor = 0.5 + (0.5 * self.effective_strength(fact))
            fused *= strength_factor

            results.append(RecallResult(fact, dense_scores.get(fact.id, 0.0), sparse_scores.get(fact.id, 0.0), fused, graph_scores.get(fact.id, 0.0)))

        results.sort(key=lambda r: r.fused_score, reverse=True)
        results = results[:limit]

        if not reinforce: return results

        refreshed: list[RecallResult] = []
        for result in results:
            latest = await self.reinforce(result.fact.id)
            if latest:
                refreshed.append(RecallResult(latest, result.dense_score, result.sparse_score, result.fused_score, result.graph_score))
        return refreshed

    # ------------------------------------------------------------------
    # Forgetting and reinforcement (Async)
    # ------------------------------------------------------------------
    def effective_strength(self, fact: Fact, at: Optional[float] = None) -> float:
        timestamp = self.clock.now() if at is None else at
        elapsed = max(0.0, timestamp - fact.last_accessed)
        decay = 2.0 ** (-elapsed / self.forgetting_half_life)
        return max(0.0, fact.strength * decay)

    async def reinforce(self, fact_id: str, amount: Optional[float] = None) -> Optional[Fact]:
        fact = await self.get_fact(fact_id)
        if not fact or not self.db: return None
        reinforcement = self.recall_reinforcement if amount is None else amount
        new_strength = min(1.0, self.effective_strength(fact) + reinforcement)

        await self.db.execute("UPDATE facts SET strength=?, last_accessed=? WHERE id=?", (new_strength, self.clock.now(), fact.id))
        await self.db.commit()
        return await self.get_fact(fact.id)


# ============================================================================
# 6. VERIFICATION HARNESS (Fully Async)
# ============================================================================

async def show_active_uses(engine: MnēmonikEvokeAsync) -> None:
    for fact in await engine.get_active_facts(predicate="uses"):
        print(f"    ACTIVE: ({fact.subject} - {fact.predicate} - {fact.object}) [Scope: {fact.scope}]")

async def run_verification() -> None:
    print("=" * 79)
    print("MNĒMONIKEVOKE v3.1 (FTS5) ASYNC VERIFICATION SUITE")
    print("=" * 79)

    clock = Clock(t0=1_700_000_000.0)

    api_key = os.environ.get("OPENAI_API_KEY", "")
    if api_key:
        print("[*] OpenAI API Key detected. Using AsyncOpenAI providers.")
        embedder = AsyncOpenAIEmbedder(api_key=api_key)
        llm = AsyncOpenAILLM(api_key=api_key)
    else:
        print("[*] No API Key. Using deterministic Async Mocks for local testing.")
        embedder = AsyncHashingEmbedder()
        llm = AsyncHeuristicLLM()

    engine = MnēmonikEvokeAsync(clock=clock, embedder=embedder, llm=llm)
    await engine.setup()

    print("\n[1] Initial ingestion")
    inputs = [
        "I use Rust for the backend.",
        "I use Rust for audio plugins.",
        "I use Postgres for storage."
    ]

    for text in inputs:
        for operation, fact in await engine.observe(text, session="init"):
            print(f"  {operation}: ({fact.subject}, {fact.predicate}, {fact.object}) scope={fact.scope}")

    print("\n[2] Initial active state")
    await show_active_uses(engine)

    print("\n[3] Advance virtual time by two days")
    clock.advance(2 * DAY)

    print("\n[4] Backend migration: Rust -> Mojo")
    migration = "We're shifting the entire backend from Rust to Mojo because we need tighter SIMD."

    for operation, fact in await engine.observe(migration, session="migration_v2"):
        print(f"  {operation}: ({fact.subject}, {fact.predicate}, {fact.object}) supersedes={fact.supersedes}")

    print("\n[5] Active state after migration")
    await show_active_uses(engine)

    print("\n[6] Hybrid retrieval test (Now using FTS5 for sparse!)")
    results = await engine.recall("backend technology", limit=5, reinforce=False)
    for result in results:
        print(f"  fused={result.fused_score:.4f} dense={result.dense_score:.4f} "
              f"sparse(BM25)={result.sparse_score:.4f} graph={result.graph_score:.4f} "
              f"fact=({result.fact.subject}, {result.fact.predicate}, {result.fact.object})")

    await engine.close()
    print("\nPASS: all FTS5 asynchronous verification checks succeeded.")

if __name__ == "__main__":
    asyncio.run(run_verification())

Overwriting mnemonik_v31_full.py


In [ ]:
%%writefile mnemonik_v31_tests_colab.py
"""Real tests for MNEMONIKEVOKE v3.1 (fixed local copy).

Covers the fusion-calibration fix, the N+1 fix, the embedding-dimension
guard, and the core engine behaviors the demo harness only prints about.
stdlib only. Run: python3 mnemonik_v31_tests.py
"""

from __future__ import annotations

import asyncio
import importlib.util
import sys

import os
MODULE_PATH = os.path.join(os.getcwd(), "mnemonik_v31_full.py")

spec = importlib.util.spec_from_file_location("mnem", MODULE_PATH)
mnem = importlib.util.module_from_spec(spec)
sys.modules["mnem"] = mnem  # dataclasses resolves annotations via sys.modules
spec.loader.exec_module(mnem)

DAY = 86_400.0
RRF_K = 60.0


async def make_engine(**kw):
    clock = mnem.Clock(t0=1_700_000_000.0)
    engine = mnem.MnēmonikEvokeAsync(
        clock=clock,
        embedder=mnem.AsyncHashingEmbedder(),
        llm=mnem.AsyncHeuristicLLM(),
        **kw,
    )
    await engine.setup()
    return engine, clock


# ---------------------------------------------------------------- ingest

async def test_ingest_and_active():
    engine, _ = await make_engine()
    await engine.observe("I use Rust for the backend.")
    await engine.observe("I use Postgres for storage.")
    active = await engine.get_active_facts()
    assert len(active) == 2, f"expected 2 active facts, got {len(active)}"
    await engine.close()


async def test_noop_on_duplicate():
    engine, clock = await make_engine()
    await engine.observe("I use Rust for the backend.")
    clock.advance(DAY)
    ops = await engine.observe("I use Rust for the backend.")
    assert ops and all(op == "NOOP" for op, _ in ops), ops
    assert len(await engine.get_active_facts()) == 1
    await engine.close()


async def test_supersession_retires_old_fact():
    engine, clock = await make_engine()
    await engine.observe("I use Rust for the backend.")
    clock.advance(2 * DAY)
    ops = await engine.observe("We're shifting the entire backend from Rust to Mojo.")
    new = [f for op, f in ops if op == "SUPERSEDE"]
    assert new, f"expected SUPERSEDE, got {[op for op, _ in ops]}"
    old = await engine.get_fact(new[0].supersedes)
    assert old is not None and old.valid_to == clock.now()
    active = await engine.get_active_facts(predicate="uses", scope="backend")
    assert [f.subject for f in active] == ["Mojo"], [f.subject for f in active]
    await engine.close()


# ---------------------------------------------------------------- fusion

async def _engine_with_heavy_graph(n_dups=200):
    engine, clock = await make_engine()
    now = clock.now()
    for i in range(n_dups):
        await engine.insert_fact(mnem.Fact(
            id=f"dup_{i}", subject="Alpha", predicate="uses", object="Beta",
            scope="t", valid_from=now, valid_to=None, supersedes=None,
            created_at=now, strength=1.0, last_accessed=now, provenance="test",
        ))
    await engine._rebuild_indexes()
    return engine


async def test_fused_score_bounded_despite_huge_graph():
    # 200 duplicate facts -> edge (Alpha,Beta,uses) weight = 200.
    # Old code added 0.001 * graph_score (unbounded); new code fuses ranks.
    engine = await _engine_with_heavy_graph()
    results = await engine.recall("Alpha", limit=5)
    assert len(results) == 5
    bound = (1.0 + 1.0 + 0.5) / (RRF_K + 1)
    for r in results:
        assert r.fused_score <= bound + 1e-9, (
            f"fused_score {r.fused_score:.4f} exceeds bound {bound:.4f} "
            f"(graph_score was {r.graph_score})"
        )
    await engine.close()


async def test_fusion_weights_zero_graph():
    engine = await _engine_with_heavy_graph()
    results = await engine.recall("Alpha", limit=5, fusion_weights=(1.0, 1.0, 0.0))
    bound = (1.0 + 1.0 + 0.0) / (RRF_K + 1)
    for r in results:
        assert r.fused_score <= bound + 1e-9, r.fused_score
    await engine.close()


async def test_fusion_weights_negative_raises():
    engine, _ = await make_engine()
    await engine.observe("I use Rust for the backend.")
    try:
        await engine.recall("backend", fusion_weights=(1.0, 1.0, -0.5))
    except ValueError:
        await engine.close()
        return
    await engine.close()
    raise AssertionError("expected ValueError for negative fusion weight")


async def test_recall_result_shape_and_limit():
    engine, _ = await make_engine()
    await engine.observe("I use Rust for the backend.")
    await engine.observe("I use Postgres for storage.")
    results = await engine.recall("backend", limit=1)
    assert len(results) == 1
    r = results[0]
    assert isinstance(r.fact, mnem.Fact)
    assert r.fused_score >= 0
    assert r.dense_score is not None and r.sparse_score is not None
    await engine.close()


# ---------------------------------------------------------------- retrieval internals

async def test_dense_rank_single_query():
    # N+1 fix: one batched vectors fetch, not one per candidate.
    # Note: aiosqlite's execute() is a plain method returning an awaitable
    # cursor, so the spy must be sync too.
    engine, _ = await make_engine()
    try:
        await engine.observe("I use Rust for the backend.")
        await engine.observe("I use Postgres for storage.")
        await engine.observe("I use Mojo for simd kernels.")
        candidates = await engine.get_active_facts()

        calls = []
        orig_execute = engine.db.execute

        def spy(sql, params=None):
            calls.append(sql)
            return orig_execute(sql, params or ())

        engine.db.execute = spy
        try:
            ranked = await engine._dense_rank("backend", candidates)
        finally:
            engine.db.execute = orig_execute
        vec_calls = [s for s in calls if "FROM vectors" in s]
        assert len(vec_calls) == 1, f"expected 1 vectors query, got {len(vec_calls)}"
        assert len(ranked) == 3
    finally:
        await engine.close()


async def test_dim_mismatch_raises():
    engine, _ = await make_engine()
    await engine.observe("I use Rust for the backend.")
    candidates = await engine.get_active_facts()
    engine.embedder = mnem.AsyncHashingEmbedder(dim=32)  # stored vectors are 64-dim
    try:
        await engine._dense_rank("backend", candidates)
    except ValueError:
        await engine.close()
        return
    await engine.close()
    raise AssertionError("expected ValueError on embedding dimension mismatch")


async def test_fts5_sparse_hit():
    engine, _ = await make_engine()
    for t in [
        "I use zxqvalve for testing.",
        "I use Rust for the backend.",
        "I use Postgres for storage.",
        "I use Mojo for simd kernels.",
        "I use Sqlite for local cache.",
    ]:
        await engine.observe(t)
    candidates = await engine.get_active_facts()
    ranked = await engine._sparse_rank("zxqvalve", candidates)
    assert ranked, "expected at least one sparse hit"
    top_fact, top_score = ranked[0]
    assert top_fact.subject == "zxqvalve", top_fact.subject
    assert top_score > 0, f"expected positive BM25-derived score, got {top_score}"
    await engine.close()


# ---------------------------------------------------------------- decay / reinforce

async def test_decay_and_reinforce():
    engine, clock = await make_engine()
    await engine.observe("I use Rust for the backend.")
    fact = (await engine.get_active_facts())[0]
    s0 = engine.effective_strength(fact)
    clock.advance(7 * DAY)  # one half-life
    s1 = engine.effective_strength(fact)
    assert abs(s1 - s0 * 0.5) < 1e-9, (s0, s1)
    refreshed = await engine.reinforce(fact.id)
    assert refreshed.strength > s1
    assert refreshed.last_accessed == clock.now()
    assert engine.effective_strength(refreshed) == refreshed.strength  # no elapsed decay
    await engine.close()


TESTS = [
    ("ingest_and_active", test_ingest_and_active),
    ("noop_on_duplicate", test_noop_on_duplicate),
    ("supersession_retires_old_fact", test_supersession_retires_old_fact),
    ("fused_score_bounded_despite_huge_graph", test_fused_score_bounded_despite_huge_graph),
    ("fusion_weights_zero_graph", test_fusion_weights_zero_graph),
    ("fusion_weights_negative_raises", test_fusion_weights_negative_raises),
    ("recall_result_shape_and_limit", test_recall_result_shape_and_limit),
    ("dense_rank_single_query", test_dense_rank_single_query),
    ("dim_mismatch_raises", test_dim_mismatch_raises),
    ("fts5_sparse_hit", test_fts5_sparse_hit),
    ("decay_and_reinforce", test_decay_and_reinforce),
]


async def main():
    failed = 0
    for name, fn in TESTS:
        try:
            await fn()
            print(f"  PASS  {name}")
        except Exception as e:  # noqa: BLE001 - test runner reports everything
            failed += 1
            print(f"  FAIL  {name}: {type(e).__name__}: {e}")
    print(f"\n{len(TESTS) - failed}/{len(TESTS)} passed")
    return 1 if failed else 0


if __name__ == "__main__":
    sys.exit(asyncio.run(main()))

Overwriting mnemonik_v31_tests_colab.py


In [ ]:
import re
import subprocess
import sys

proc = subprocess.run(
    [sys.executable, "mnemonik_v31_tests_colab.py"],
    capture_output=True,
    text=True,
)
print(proc.stdout, end="")
if proc.stderr:
    print(proc.stderr, end="", file=sys.stderr)

m = re.search(r"(\d+)/(\d+) passed", proc.stdout)
passed, total = (int(m.group(1)), int(m.group(2))) if m else (0, 0)
ok = proc.returncode == 0 and m is not None and total > 0 and passed == total
verdict = "PASS" if ok else "FAIL"
print(f"MN\u0112MONIKEVOKE v3.1 COLAB VERIFICATION: {verdict} \u2014 {passed}/{total} tests passed")

  PASS  ingest_and_active
  PASS  noop_on_duplicate
  PASS  supersession_retires_old_fact
  PASS  fused_score_bounded_despite_huge_graph
  PASS  fusion_weights_zero_graph
  PASS  fusion_weights_negative_raises
  PASS  recall_result_shape_and_limit
  PASS  dense_rank_single_query
  PASS  dim_mismatch_raises
  PASS  fts5_sparse_hit
  PASS  decay_and_reinforce

11/11 passed
MNĒMONIKEVOKE v3.1 COLAB VERIFICATION: PASS — 11/11 tests passed
